In [1]:
from pathlib import Path
import hashlib
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from cycler import cycler
from IPython.display import display
from split_policy import assign_household_split, SPLIT_SEED, TEST_FRACTION

ROOT = Path.cwd()
SOURCE = ROOT / 'inputs' / 'households.xlsx'
INTERMEDIATE = ROOT / 'intermediaries'
OUTPUT = ROOT / 'final_outputs' / 'EDA'
FIGURES = OUTPUT / 'figures'
TRAIN_BOOK = INTERMEDIATE / 'EDA_training.xlsx'
TEST_BOOK = INTERMEDIATE / 'EDA_heldout_test.xlsx'
FINAL_BOOK = OUTPUT / 'Household_EDA_results.xlsx'
for folder in [INTERMEDIATE, OUTPUT, FIGURES]: folder.mkdir(parents=True, exist_ok=True)

FIELDS = ['hnum', 'gta01hhld', 'dwelltype', 'rec-mail', 'n-call', 'tripday',
          'nperson', 'nvehicle', 'nlicence', 'nempft', 'nemppt', 'nemphome',
          'nstudent', 'nhhldtrip', 'expf']
REGIONS = {'Toronto': (1, 481), 'Durham': (501, 765),
           'York': (1001, 1353), 'Peel': (1501, 1753)}
SIZE_TOP, DRIVER_TOP, VEHICLE_TOP = 5, 3, 3
MIN_CELL_N = 30
BLUE, GOLD, TEAL, CORAL, INK = '#2673B8', '#E9A23B', '#2E9B8A', '#D96B55', '#253746'
PALE = '#F2EFE8'
WARM_HEAT = LinearSegmentedColormap.from_list(
    'trip_warm', ['#FFF7E8','#FBE0A3','#F2B75F','#D9843D','#A95532'])
TEAL_HEAT = LinearSegmentedColormap.from_list(
    'trip_teal', ['#F2FBF8','#C6E8DF','#78CDBB','#238E82'])
CORRELATION_HEAT = LinearSegmentedColormap.from_list(
    'trip_correlation', ['#237D86','#C6E3E0','#FFF8EA','#F7C873','#C56B3D'])
WARM_HEAT.set_bad(PALE)
TEAL_HEAT.set_bad(PALE)
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#FFFCF7',
    'axes.edgecolor':INK,'axes.labelcolor':INK,'text.color':INK,
    'xtick.color':INK,'ytick.color':INK,'font.size':10,
    'axes.spines.top':False,'axes.spines.right':False,
    'axes.prop_cycle':cycler(color=[BLUE,GOLD,TEAL,CORAL,'#8A914B']),
    'pdf.fonttype':42})

def write_workbook(path, sheets):
    temporary = path.with_name(path.stem + '_building.xlsx')
    names = [name[:31] for name in sheets]
    assert len(names) == len(set(names))
    with pd.ExcelWriter(temporary, engine='openpyxl') as writer:
        for name, frame in sheets.items():
            frame.to_excel(writer, sheet_name=name[:31], index=False)
            sheet = writer.sheets[name[:31]]
            sheet.freeze_panes = 'A2'
            sheet.auto_filter.ref = sheet.dimensions
    temporary.replace(path)

def grouped_rates(frame, field):
    rows = []
    for level, block in frame.groupby(field, sort=True, observed=True, dropna=False):
        w = block['expf'].to_numpy(float)
        y = block['nhhldtrip'].to_numpy(float)
        rows.append({'variable':field,'level':str(level),'n':len(block),
                     'mean_trips':y.mean(),'weighted_mean_trips':np.average(y,weights=w),
                     'zero_pct':100*np.mean(y==0)})
    return pd.DataFrame(rows)
FIELD_LABELS = {
    'nperson':'Household size', 'nvehicle':'Number of vehicles',
    'nlicence':'Number of licensed drivers', 'nempft':'Number of full-time workers',
    'nemppt':'Number of part-time workers', 'nemphome':'Number of home workers',
    'nstudent':'Number of students', 'workers':'Total number of workers',
    'nhhldtrip':'Recorded weekday household trips', 'expf':'Household expansion factor',
    'dwelling':'Dwelling type', 'weekday':'Survey weekday', 'region':'Region',
}
def label_panels(axes):
    for letter, ax in zip('abcdefghijklmnopqrstuvwxyz', np.asarray(axes).flat):
        title = ax.get_title()
        if title:
            title = title.replace('Number of ', '').replace('Total number of workers', 'Total workers')
            ax.set_title('')
            ax.set_title(f'({letter}) {title}', loc='left', fontsize=10)
        else:
            ax.text(0, 1.03, f'({letter})', transform=ax.transAxes,
                    ha='left', va='bottom', fontweight='bold')

print(f'Python interpreter: {sys.executable}')
print(f'NumPy version: {np.__version__}')


Python interpreter: /home/mst_lab/geo_env/bin/python
NumPy version: 2.4.6


In [2]:
source = pd.read_excel(SOURCE, sheet_name='2006-hhld', usecols='B:P', dtype=object, engine='openpyxl')
assert source.columns.tolist() == FIELDS
source.insert(0, 'source_excel_row', np.arange(len(source)) + 2)
for field in FIELDS:
    if field != 'rec-mail':
        source[field] = pd.to_numeric(source[field], errors='raise')
        assert np.isfinite(source[field]).all(), field
        if field != 'expf':
            assert source[field].mod(1).eq(0).all(), field
            source[field] = source[field].astype('int64')
assert source['hnum'].is_unique
assert source['dwelltype'].isin([1,2,3,9]).all()
assert source['tripday'].isin([1,2,3,4,5]).all()
assert source['nperson'].between(1,99).all()
assert source['nvehicle'].between(0,99).all()
assert source['nhhldtrip'].between(0,99).all()
assert source['expf'].gt(0).all()
for field in ['nlicence','nempft','nemppt','nemphome','nstudent']:
    assert source[field].between(0,9).all(), field
assert source['nlicence'].le(source['nperson']).all()
assert (source['nempft']+source['nemppt']).le(source['nperson']).all()
assert source['nemphome'].le(source['nempft']+source['nemppt']).all()
assert source['nstudent'].le(source['nperson']).all()


# Generate the common split directly from source household IDs on every run.
print(f'Split seed = {SPLIT_SEED}; test fraction = {TEST_FRACTION:.0%}')
split = assign_household_split(source['hnum'])
assert split['hnum'].is_unique and set(split['split']) == {'train','test'}
assert set(split['hnum']) == set(source['hnum'])
source = source.merge(split, on='hnum', validate='one_to_one', sort=False)
source['exclude_vehicles_10plus'] = source['nvehicle'].ge(10)
source['exclude_unknown_dwelling'] = source['dwelltype'].eq(9)
excluded = source.loc[source['exclude_vehicles_10plus'] | source['exclude_unknown_dwelling']].copy()
excluded['reason'] = [', '.join(reason for flag,reason in
    [(v,'vehicles >=10'),(d,'dwelling 9 Unknown')] if flag) for v,d in
    zip(excluded['exclude_vehicles_10plus'],excluded['exclude_unknown_dwelling'])]
eligible = source.loc[~(source['exclude_vehicles_10plus'] | source['exclude_unknown_dwelling'])]
train = eligible.loc[eligible['split'].eq('train'), ['source_excel_row']+FIELDS+['split']].copy().reset_index(drop=True)
heldout = eligible.loc[eligible['split'].eq('test'), ['source_excel_row']+FIELDS+['split']].copy().reset_index(drop=True)
assert len(train)+len(heldout)+len(excluded)==len(source)
assert set(train['hnum']).isdisjoint(set(heldout['hnum']))

# Store the test records separately, then release them before EDA calculations.
write_workbook(TEST_BOOK, {'Test': heldout})
source_n, train_n, test_n, excluded_n = len(source),len(train),len(heldout),len(excluded)
source_hash = hashlib.sha256(SOURCE.read_bytes()).hexdigest()
del source, eligible, heldout

train['workers'] = train['nempft'] + train['nemppt']
train['dwelling'] = train['dwelltype'].map({1:'House',2:'Apartment',3:'Townhouse'})
train['weekday'] = train['tripday'].map(dict(enumerate(
    ['Monday','Tuesday','Wednesday','Thursday','Friday'],start=1)))
train['region'] = ''
for region,(low,high) in REGIONS.items():
    train.loc[train['gta01hhld'].between(low,high),'region'] = region
assert train['region'].ne('').all()
train['size_band'] = train['nperson'].clip(upper=SIZE_TOP)
train['driver_band'] = train['nlicence'].clip(upper=DRIVER_TOP)
train['vehicle_band'] = train['nvehicle'].clip(upper=VEHICLE_TOP)

quality = pd.DataFrame([
    {'check':'Source schema and numeric ranges','result':'Passed'},
    {'check':'Unique household IDs','result':'Passed'},
    {'check':'Within-household count consistency','result':'Passed'},
    {'check':'Seeded split IDs match source IDs','result':'Passed'},
    {'check':'Train/test IDs disjoint','result':'Passed'},
    {'check':'Source workbook SHA256','result':source_hash},
])
flow = pd.DataFrame([
    {'stage':'Supplied source households','households':source_n},
    {'stage':'Excluded: vehicles >=10','households':int(excluded['exclude_vehicles_10plus'].sum())},
    {'stage':'Excluded: dwelling 9 Unknown','households':int(excluded['exclude_unknown_dwelling'].sum())},
    {'stage':'Unique excluded','households':excluded_n},
    {'stage':'Common eligible training','households':train_n},
    {'stage':'Common eligible held-out test','households':test_n},
])
display(flow)
print('Held-out records saved separately. No test outcomes enter subsequent EDA cells.')

Split seed = 42; test fraction = 30%


,stage,households
0,Supplied source households,93307
1,Excluded: vehicles >=10,33
2,Excluded: dwelling 9 Unknown,9
3,Unique excluded,42
4,Common eligible training,65287
5,Common eligible held-out test,27978


Held-out records saved separately. No test outcomes enter subsequent EDA cells.


In [3]:
numeric_fields = ['nperson','nvehicle','nlicence','nempft','nemppt',
                  'nemphome','nstudent','workers','nhhldtrip','expf']
numeric = pd.DataFrame([{'variable':field,'n':len(train),
    'mean':train[field].mean(),'sd':train[field].std(),'median':train[field].median(),
    'p25':train[field].quantile(.25),'p75':train[field].quantile(.75),
    'minimum':train[field].min(),'maximum':train[field].max(),
    'zero_n':int(train[field].eq(0).sum())} for field in numeric_fields])
trip_counts = train['nhhldtrip'].value_counts().sort_index()
trip_distribution = pd.DataFrame({'trips':trip_counts.index,'households':trip_counts.values})
trip_distribution['percent'] = 100*trip_distribution['households']/len(train)
categories = pd.concat([grouped_rates(train,field) for field in
    ['size_band','vehicle_band','driver_band','workers','nstudent','dwelling','weekday','region']],
    ignore_index=True)
display(numeric)
display(categories.head(16))

,variable,n,mean,sd,median,p25,p75,minimum,maximum,zero_n
0,nperson,65287,2.736915,1.417725,2.00,2.00,4.00,1.00,16.00,0
1,nvehicle,65287,1.376430,0.929003,1.00,1.00,2.00,0.00,9.00,10708
2,nlicence,65287,1.729977,0.983994,2.00,1.00,2.00,0.00,9.00,6261
3,nempft,65287,1.112993,0.926035,1.00,0.00,2.00,0.00,7.00,19512
4,nemppt,65287,0.237413,0.510708,0.00,0.00,0.00,0.00,5.00,52090
5,nemphome,65287,0.109149,0.356132,0.00,0.00,0.00,0.00,7.00,59058
6,nstudent,65287,0.633281,0.951703,0.00,0.00,1.00,0.00,9.00,40484
7,workers,65287,1.350407,1.047334,1.00,1.00,2.00,0.00,9.00,16063
8,nhhldtrip,65287,5.642364,4.548931,4.00,2.00,8.00,0.00,55.00,7274
9,expf,65287,19.391051,2.514385,19.35,17.67,20.99,10.83,28.06,0


,variable,level,n,mean_trips,weighted_mean_trips,zero_pct
0,size_band,1,13431,2.036259,2.035788,30.585958
1,size_band,2,20319,4.352035,4.348931,12.815591
2,size_band,3,12158,6.429840,6.419431,2.796513
3,size_band,4,12185,8.313500,8.298296,1.247435
4,size_band,5,7194,10.164165,10.151932,0.973033
5,vehicle_band,0,10708,2.353941,2.363725,34.039970
6,vehicle_band,1,26880,4.852753,4.855798,10.740327
7,vehicle_band,2,21814,7.312139,7.308624,3.103512
8,vehicle_band,3,5885,9.042991,9.054557,1.104503
9,driver_band,0,6261,1.725124,1.738933,45.951126


In [4]:
def cell_grid(row_field, col_field, row_levels, col_levels):
    grouped = train.groupby([row_field,col_field],observed=True).agg(
        n=('hnum','size'), sd_trips=('nhhldtrip','std'), trips=('nhhldtrip','sum'), weighted_trips=('nhhldtrip',
            lambda x: np.nan), weight_sum=('expf','sum')).reset_index()
    # Weighted trip totals are calculated directly to keep the denominator explicit.
    weighted = train.assign(weighted_trips=train['nhhldtrip']*train['expf']).groupby(
        [row_field,col_field],observed=True)['weighted_trips'].sum().reset_index()
    grouped = grouped.drop(columns='weighted_trips').merge(weighted,on=[row_field,col_field])
    grid = pd.MultiIndex.from_product([row_levels,col_levels],names=[row_field,col_field]).to_frame(index=False)
    result = grid.merge(grouped,on=[row_field,col_field],how='left')
    result['n'] = result['n'].fillna(0).astype(int)
    result['mean_trips'] = result['trips']/result['n'].replace(0,np.nan)
    result['cv'] = result['sd_trips']/result['mean_trips'].replace(0,np.nan)
    result['weighted_mean_trips'] = result['weighted_trips']/result['weight_sum']
    result['sparse_under_30'] = result['n'].lt(MIN_CELL_N)
    return result.drop(columns=['trips','weighted_trips'])

size_driver = cell_grid('size_band','driver_band',range(1,SIZE_TOP+1),range(DRIVER_TOP+1))
size_driver['structurally_possible'] = size_driver['size_band'].eq(SIZE_TOP) | size_driver['driver_band'].le(size_driver['size_band'])
size_vehicle = cell_grid('size_band','vehicle_band',range(1,SIZE_TOP+1),range(VEHICLE_TOP+1))
assert size_driver['n'].sum() == len(train) == size_vehicle['n'].sum()
display(size_driver)

,size_band,driver_band,n,sd_trips,weight_sum,mean_trips,cv,weighted_mean_trips,sparse_under_30,structurally_possible
0,1,0,4080,1.334687,79010.86,1.094853,1.219056,1.101737,False,True
1,1,1,9351,1.843082,181686.30,2.447011,0.753197,2.441983,False,True
2,1,2,0,NaN,NaN,NaN,NaN,NaN,True,False
3,1,3,0,NaN,NaN,NaN,NaN,NaN,True,False
4,2,0,1473,2.123991,28706.22,2.105227,1.008913,2.129584,False,True
5,2,1,5519,2.887526,107092.69,3.811560,0.757571,3.816668,False,True
6,2,2,13327,3.011697,257004.45,4.824191,0.624291,4.818613,False,True
7,2,3,0,NaN,NaN,NaN,NaN,NaN,True,False
8,3,0,401,2.413958,7822.98,3.982544,0.606135,3.982729,False,True
9,3,1,2218,3.533475,43293.05,5.510821,0.641189,5.494354,False,True


In [5]:
zone_support = train.groupby('gta01hhld').agg(
    households=('hnum','size'),observed_sample_trips=('nhhldtrip','sum'),
    expansion_factor_sum=('expf','sum')).reset_index()
correlation_fields = ['nperson','nvehicle','nlicence','nempft','nemppt','nemphome','nstudent','workers','nhhldtrip']
correlations = train[correlation_fields].corr(method='pearson').reset_index().rename(columns={'index':'variable'})
weighting = pd.DataFrame([
    {'measure':'Training household mean trips','unweighted':train['nhhldtrip'].mean(),
     'expansion_weighted':np.average(train['nhhldtrip'],weights=train['expf'])},
    {'measure':'Training zero-trip share','unweighted':train['nhhldtrip'].eq(0).mean(),
     'expansion_weighted':np.average(train['nhhldtrip'].eq(0),weights=train['expf'])},
])

fig,ax=plt.subplots(figsize=(9,4.5))
ax.bar(train['nhhldtrip'].clip(upper=25).value_counts().sort_index().index,
       train['nhhldtrip'].clip(upper=25).value_counts().sort_index().values,color=BLUE)
ax.set(xlabel='Recorded weekday household trips (25+ grouped)',ylabel='Households')
fig.tight_layout();fig.savefig(FIGURES/'EDA_01_trip_distribution.png',dpi=220);plt.close(fig)

fig,axes=plt.subplots(1,3,figsize=(12,3.8))
for ax,field,title in zip(axes,['size_band','vehicle_band','driver_band'],
                          ['Household size','Available vehicles','Licensed drivers']):
    block=categories.loc[categories['variable'].eq(field)]
    ax.plot(block['level'],block['mean_trips'],marker='o',color=BLUE)
    ax.set(xlabel=title,ylabel='Mean weekday household trips',title=title)
label_panels(axes)
fig.tight_layout();fig.savefig(FIGURES/'EDA_02_group_rates.png',dpi=220);plt.close(fig)

heat=size_driver.pivot(index='size_band',columns='driver_band',values='mean_trips')
fig,ax=plt.subplots(figsize=(7,4.5));im=ax.imshow(heat,cmap=WARM_HEAT,aspect='auto')
ax.set(xticks=range(len(heat.columns)),xticklabels=[f'{x}+' if x==DRIVER_TOP else str(x) for x in heat.columns],
       yticks=range(len(heat.index)),yticklabels=[f'{x}+' if x==SIZE_TOP else str(x) for x in heat.index],
       xlabel='Number of licensed drivers',ylabel='Household size')
for r in range(len(heat.index)):
    for c in range(len(heat.columns)):
        if np.isfinite(heat.iloc[r,c]):ax.text(c,r,f'{heat.iloc[r,c]:.1f}',ha='center',va='center')
fig.colorbar(im,ax=ax,label='Mean weekday household trips');fig.tight_layout()
fig.savefig(FIGURES/'EDA_03_size_driver_cells.png',dpi=220);plt.close(fig)
display(weighting)

,measure,unweighted,expansion_weighted
0,Training household mean trips,5.642364,5.638513
1,Training zero-trip share,0.111416,0.110763


In [6]:
# Detailed profiles of the same eligible training sample.
count_fields = ['nperson','nvehicle','nlicence','nempft','nemppt','nemphome','nstudent','workers']
category_fields = ['dwelling','weekday','region']
display_caps = {'nperson':5,'nvehicle':3,'nlicence':3,'nempft':4,'nemppt':3,
                'nemphome':3,'nstudent':4,'workers':4}
detail_tables = {}

def rate_table(frame, field, grouped=False):
    values = frame[field].clip(upper=display_caps[field]) if grouped and field in display_caps else frame[field]
    data = frame.assign(_level=values, _weighted_trips=frame['nhhldtrip']*frame['expf'],
                        _weighted_zero=frame['nhhldtrip'].eq(0)*frame['expf'])
    out = data.groupby('_level', sort=True, observed=True).agg(
        n=('hnum','size'), mean_trips=('nhhldtrip','mean'), sd_trips=('nhhldtrip','std'),
        median_trips=('nhhldtrip','median'), zero_n=('nhhldtrip',lambda s: int(s.eq(0).sum())),
        weight_sum=('expf','sum'), weighted_trips=('_weighted_trips','sum'),
        weighted_zero=('_weighted_zero','sum')).reset_index().rename(columns={'_level':'level'})
    out['zero_pct'] = 100*out['zero_n']/out['n']
    out['weighted_mean_trips'] = out['weighted_trips']/out['weight_sum']
    out['weighted_zero_pct'] = 100*out['weighted_zero']/out['weight_sum']
    if field in category_fields:
        order = {'dwelling':['House','Apartment','Townhouse'],
                 'weekday':['Monday','Tuesday','Wednesday','Thursday','Friday'],
                 'region':['Toronto','Durham','York','Peel']}[field]
        out['_sort'] = out['level'].map({level:i for i,level in enumerate(order)})
        out = out.sort_values('_sort').drop(columns='_sort')
    out['variable'] = field
    out['display_grouped'] = grouped and field in display_caps
    out['level_label'] = out['level'].astype(str)
    if grouped and field in display_caps:
        out.loc[out['level'].eq(display_caps[field]),'level_label'] += '+'
    return out[['variable','level','level_label','display_grouped','n','mean_trips',
                'sd_trips','median_trips','zero_n','zero_pct','weight_sum',
                'weighted_mean_trips','weighted_zero_pct']]

detail_tables['ExactGroupRates'] = pd.concat(
    [rate_table(train,f) for f in count_fields+category_fields],ignore_index=True)
detail_tables['DisplayGroupRates'] = pd.concat(
    [rate_table(train,f,True) for f in count_fields+category_fields],ignore_index=True)
detail_tables['CategoricalFrequencies'] = pd.concat([
    train.groupby(field,observed=True).agg(n=('hnum','size'),weight_sum=('expf','sum'))
         .reset_index().rename(columns={field:'level'}).assign(variable=field)
    for field in category_fields+['size_band','vehicle_band','driver_band']],ignore_index=True)
detail_tables['CategoricalFrequencies']['sample_pct'] = (
    100*detail_tables['CategoricalFrequencies']['n']/len(train))

detail_tables['NumericDetail'] = pd.DataFrame([{
    'variable':f,'n':len(train),'mean':train[f].mean(),'sd':train[f].std(),
    'se_mean':train[f].sem(),'coefficient_of_variation':train[f].std()/train[f].mean() if train[f].mean() else np.nan,
    'minimum':train[f].min(),'p05':train[f].quantile(.05),'p10':train[f].quantile(.10),
    'p25':train[f].quantile(.25),'median':train[f].median(),'p75':train[f].quantile(.75),
    'p90':train[f].quantile(.90),'p95':train[f].quantile(.95),'p99':train[f].quantile(.99),
    'maximum':train[f].max(),'zero_n':int(train[f].eq(0).sum()),
    'zero_pct':np.nan if f=='expf' else 100*train[f].eq(0).mean(),'missing_n':int(train[f].isna().sum())}
    for f in count_fields+['nhhldtrip','expf']])

def cross_cells(frame, col_field, cap):
    data = frame.assign(size=frame['nperson'].clip(upper=SIZE_TOP),
                        level=frame[col_field].clip(upper=cap),
                        weighted_trips=frame['nhhldtrip']*frame['expf'])
    cells = data.groupby(['size','level'],observed=True).agg(
        n=('hnum','size'),mean_trips=('nhhldtrip','mean'),sd_trips=('nhhldtrip','std'),
        zero_n=('nhhldtrip',lambda s: int(s.eq(0).sum())),
        weight_sum=('expf','sum'),weighted_trips=('weighted_trips','sum')).reset_index()
    grid = pd.MultiIndex.from_product([range(1,SIZE_TOP+1),range(cap+1)],
                                      names=['size','level']).to_frame(index=False)
    cells = grid.merge(cells,on=['size','level'],how='left')
    cells[['n','zero_n']] = cells[['n','zero_n']].fillna(0).astype(int)
    cells['weight_sum'] = cells['weight_sum'].fillna(0)
    cells['weighted_mean_trips'] = cells['weighted_trips']/cells['weight_sum'].replace(0,np.nan)
    cells['zero_pct'] = 100*cells['zero_n']/cells['n'].replace(0,np.nan)
    cells['sparse_under_30'] = cells['n'].lt(MIN_CELL_N)
    cells['scheme'] = f'5+ persons by {cap}+ {col_field}'
    cells['structurally_possible'] = True if col_field=='nvehicle' else (
        cells['size'].eq(SIZE_TOP) | cells['level'].le(cells['size']))
    return cells.drop(columns='weighted_trips')

detail_tables['VehicleCellsAlt'] = cross_cells(train,'nvehicle',4)
detail_tables['DriverCellsAlt'] = cross_cells(train,'nlicence',4)
all_cells = pd.concat([
    size_vehicle.rename(columns={'size_band':'size','vehicle_band':'level'}).assign(scheme='5+ persons by 3+ nvehicle'),
    detail_tables['VehicleCellsAlt'],
    size_driver.rename(columns={'size_band':'size','driver_band':'level'}).assign(scheme='5+ persons by 3+ nlicence'),
    detail_tables['DriverCellsAlt']],ignore_index=True)
adjacent = []
for scheme,block in all_cells.groupby('scheme',sort=False):
    for change,fixed in [('size','level'),('level','size')]:
        for fixed_level,line in block.groupby(fixed):
            line = line.sort_values(change)
            for (_,left),(_,right) in zip(line.iloc[:-1].iterrows(),line.iloc[1:].iterrows()):
                adjacent.append({'scheme':scheme,'changing_dimension':change,
                    'fixed_dimension':fixed,'fixed_level':fixed_level,
                    'from_level':int(left[change]),'to_level':int(right[change]),
                    'n_from':int(left['n']),'n_to':int(right['n']),
                    'mean_difference':right['mean_trips']-left['mean_trips'],
                    'weighted_mean_difference':right['weighted_mean_trips']-left['weighted_mean_trips'],
                    'both_supported_30':min(left['n'],right['n'])>=MIN_CELL_N})
detail_tables['AdjacentCellDifferences'] = pd.DataFrame(adjacent)

frequency = []
for field,cap in [('nperson',5),('nvehicle',3)]:
    data = train.assign(level=train[field].clip(upper=cap),trips=train['nhhldtrip'].clip(upper=20))
    counts = data.groupby(['level','trips']).size().rename('n')
    grid = pd.MultiIndex.from_product([range(1 if field=='nperson' else 0,cap+1),
                                       range(21)],names=['level','trips'])
    block = counts.reindex(grid,fill_value=0).reset_index()
    block['variable']=field
    block['level_label']=block['level'].astype(str)
    block.loc[block['level'].eq(cap),'level_label']+='+'
    block['trip_label']=block['trips'].astype(str)
    block.loc[block['trips'].eq(20),'trip_label']='20+'
    block['group_n']=block.groupby('level')['n'].transform('sum')
    frequency.append(block)
detail_tables['TripFrequencies'] = pd.concat(frequency,ignore_index=True)
assert all(detail_tables['TripFrequencies'].groupby('variable')['n'].sum().eq(len(train)))

predictors = count_fields
pair = []
for left in predictors+['nhhldtrip']:
    for right in predictors+['nhhldtrip']:
        pair.append({'variable_1':left,'variable_2':right,'n':len(train),
                     'pearson':train[left].corr(train[right],method='pearson')})
detail_tables['PairwiseCorrelations'] = pd.DataFrame(pair)
detail_tables['OutcomeCorrelations'] = detail_tables['PairwiseCorrelations'].loc[
    detail_tables['PairwiseCorrelations']['variable_2'].eq('nhhldtrip') &
    detail_tables['PairwiseCorrelations']['variable_1'].ne('nhhldtrip')].copy()

regional = []
for region,block in train.groupby('region',sort=True):
    cells = cross_cells(block,'nlicence',3)
    cells.insert(0,'region',region)
    cells['region_n']=len(block)
    cells['share_of_region_pct']=100*cells['n']/len(block)
    regional.append(cells)
detail_tables['RegionalDriverCells'] = pd.concat(regional,ignore_index=True)

# Pre-exclusion review uses training-assigned households only; test rows stay hidden.
pre_exclusion_train = pd.concat([train[['hnum','nvehicle','nperson','nlicence','dwelltype','nhhldtrip']],
    excluded.loc[excluded['split'].eq('train'),
                 ['hnum','nvehicle','nperson','nlicence','dwelltype','nhhldtrip']]],ignore_index=True)
assert pre_exclusion_train['hnum'].is_unique
review_counts = pre_exclusion_train['nvehicle'].value_counts().sort_index()
detail_tables['VehiclePreExclusion'] = pd.DataFrame({
    'nvehicle':review_counts.index,'training_households':review_counts.values})
detail_tables['VehiclePreExclusion']['training_share_pct'] = (
    100*detail_tables['VehiclePreExclusion']['training_households']/len(pre_exclusion_train))
vehicle_tail = pre_exclusion_train.loc[pre_exclusion_train['nvehicle'].ge(5)].copy()
vehicle_tail['vehicle_band'] = np.where(vehicle_tail['nvehicle'].ge(10),'10+','5-9')
detail_tables['VehicleTailReview'] = vehicle_tail.groupby('vehicle_band').agg(
    n=('hnum','size'),mean_size=('nperson','mean'),mean_licensed=('nlicence','mean'),
    mean_trips=('nhhldtrip','mean'),median_trips=('nhhldtrip','median'),
    house_n=('dwelltype',lambda v:int(v.eq(1).sum())),
    apartment_n=('dwelltype',lambda v:int(v.eq(2).sum())),
    townhouse_n=('dwelltype',lambda v:int(v.eq(3).sum())),
    unknown_dwelling_n=('dwelltype',lambda v:int(v.eq(9).sum()))).reset_index()

display(detail_tables['NumericDetail'])
display(detail_tables['ExactGroupRates'].head(12))
display(detail_tables['OutcomeCorrelations'])
print(f"Prepared {len(detail_tables)} additional training-only descriptive tables.")

,variable,n,mean,sd,se_mean,coefficient_of_variation,minimum,p05,p10,p25,median,p75,p90,p95,p99,maximum,zero_n,zero_pct,missing_n
0,nperson,65287,2.736915,1.417725,0.005549,0.518001,1.00,1.00,1.00,2.00,2.00,4.00,5.00,5.00,7.00,16.00,0,0.000000,0
1,nvehicle,65287,1.376430,0.929003,0.003636,0.674936,0.00,0.00,0.00,1.00,1.00,2.00,2.00,3.00,4.00,9.00,10708,16.401428,0
2,nlicence,65287,1.729977,0.983994,0.003851,0.568790,0.00,0.00,1.00,1.00,2.00,2.00,3.00,3.00,4.00,9.00,6261,9.589964,0
3,nempft,65287,1.112993,0.926035,0.003624,0.832022,0.00,0.00,0.00,0.00,1.00,2.00,2.00,3.00,4.00,7.00,19512,29.886501,0
4,nemppt,65287,0.237413,0.510708,0.001999,2.151137,0.00,0.00,0.00,0.00,0.00,0.00,1.00,1.00,2.00,5.00,52090,79.786175,0
5,nemphome,65287,0.109149,0.356132,0.001394,3.262807,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,2.00,7.00,59058,90.459050,0
6,nstudent,65287,0.633281,0.951703,0.003725,1.502813,0.00,0.00,0.00,0.00,0.00,1.00,2.00,3.00,4.00,9.00,40484,62.009282,0
7,workers,65287,1.350407,1.047334,0.004099,0.775569,0.00,0.00,0.00,1.00,1.00,2.00,3.00,3.00,4.00,9.00,16063,24.603673,0
8,nhhldtrip,65287,5.642364,4.548931,0.017803,0.806210,0.00,0.00,0.00,2.00,4.00,8.00,12.00,14.00,21.00,55.00,7274,11.141575,0
9,expf,65287,19.391051,2.514385,0.009841,0.129667,10.83,15.32,16.33,17.67,19.35,20.99,22.54,23.53,26.13,28.06,0,NaN,0


,variable,level,level_label,display_grouped,n,mean_trips,sd_trips,median_trips,zero_n,zero_pct,weight_sum,weighted_mean_trips,weighted_zero_pct
0,nperson,1,1,False,13431,2.036259,1.814588,2.0,4108,30.585958,260697.16,2.035788,30.426189
1,nperson,2,2,False,20319,4.352035,3.021879,4.0,2604,12.815591,392803.36,4.348931,12.727595
2,nperson,3,3,False,12158,6.429840,3.652005,6.0,340,2.796513,235696.23,6.419431,2.786778
3,nperson,4,4,False,12185,8.313500,4.710430,8.0,152,1.247435,236769.39,8.298296,1.251336
4,nperson,5,5,False,4865,9.818294,5.574985,9.0,51,1.048304,94680.55,9.807105,1.076092
5,nperson,6,6,False,1625,10.420308,5.704868,10.0,13,0.800000,31656.66,10.393742,0.797178
6,nperson,7,7,False,464,11.316810,6.172192,10.0,3,0.646552,9004.56,11.340768,0.597364
7,nperson,8,8,False,152,11.861842,6.777494,10.5,3,1.973684,2967.78,11.848783,1.811455
8,nperson,9,9,False,47,15.085106,7.107344,14.0,0,0.000000,918.91,15.157676,0.000000
9,nperson,10,10,False,24,16.333333,10.748778,14.0,0,0.000000,469.61,16.288878,0.000000


,variable_1,variable_2,n,pearson
8,nperson,nhhldtrip,65287,0.571830
17,nvehicle,nhhldtrip,65287,0.431731
26,nlicence,nhhldtrip,65287,0.513932
35,nempft,nhhldtrip,65287,0.442602
44,nemppt,nhhldtrip,65287,0.286946
53,nemphome,nhhldtrip,65287,0.090347
62,nstudent,nhhldtrip,65287,0.516687
71,workers,nhhldtrip,65287,0.531264


Prepared 13 additional training-only descriptive tables.


In [7]:
# Restored figure series from the original EDA, recalculated on training households.
EDA_FIGURE_NAMES = {1: 'household_trip_distribution', 2: 'household_characteristic_distributions', 3: 'household_characteristic_trip_rates', 4: 'household_size_vehicle_cells', 6: 'predictor_correlation_heatmap', 7: 'household_characteristic_trip_frequencies', 8: 'outcome_correlations', 9: 'dwelling_weekday_region_trip_rates', 10: 'household_size_driver_cells', 11: 'licensed_driver_trip_rate_profiles'}

def save_eda_figure(fig, number):
    fig.savefig(FIGURES/f'Figure_ED{number:02d}_{EDA_FIGURE_NAMES[number]}.png',dpi=240,facecolor='white',bbox_inches='tight')
    plt.close(fig)

def plot_rates(fields, number, ncols=2):
    nrows=int(np.ceil(len(fields)/ncols))
    fig,axes=plt.subplots(nrows,ncols,figsize=(5.2*ncols,3.5*nrows),squeeze=False)
    for ax,field in zip(axes.flat,fields):
        data=detail_tables['DisplayGroupRates'].query('variable == @field')
        x=np.arange(len(data))
        if field in category_fields:
            ax.bar(x,data['mean_trips'],color=BLUE)
        else:
            ax.plot(x,data['mean_trips'],marker='o',color=BLUE)
        ax.set_xticks(x,data['level_label'].astype(str),rotation=35 if field in category_fields else 0)
        ax.set(xlabel=FIELD_LABELS[field],ylabel='Mean weekday household trips',title=FIELD_LABELS[field])
        ax.set_ylim(0,max(1,data['mean_trips'].max()*1.24))
        for x0,y,n in zip(x,data['mean_trips'],data['n']):
            ax.annotate(f'n={n:,}',(x0,y),xytext=(0,7),textcoords='offset points',
                        ha='center',fontsize=7,rotation=0 if field in category_fields else 35)
    for ax in axes.flat[len(fields):]:ax.axis('off')
    label_panels(list(axes.flat)[:len(fields)])
    fig.tight_layout();save_eda_figure(fig,number)

def plot_cell_heat(cells,number,title,column_label):
    fig,axes=plt.subplots(1,2,figsize=(11,4.5))
    for ax,value,label,cmap in zip(axes,['mean_trips','n'],['Mean trips','Households'],[WARM_HEAT,TEAL_HEAT]):
        matrix=cells.pivot(index='size',columns='level',values=value)
        if 'structurally_possible' in cells:
            possible=cells.pivot(index='size',columns='level',values='structurally_possible')
            matrix=matrix.where(possible)
        image=ax.imshow(matrix,cmap=cmap,aspect='auto')
        ax.set(xticks=range(len(matrix.columns)),xticklabels=[f'{v}+' if v==3 else str(v) for v in matrix.columns],
               yticks=range(len(matrix.index)),yticklabels=[f'{v}+' if v==SIZE_TOP else str(v) for v in matrix.index],
               xlabel=column_label,ylabel='Household size')
        for r in range(len(matrix.index)):
            for c in range(len(matrix.columns)):
                val=matrix.iloc[r,c]
                if np.isfinite(val):ax.text(c,r,f'{val:,.1f}' if value=='mean_trips' else f'{val:,.0f}',
                                            ha='center',va='center',fontsize=8)
        fig.colorbar(image,ax=ax,label='Mean weekday household trips' if value=='mean_trips' else 'Households',shrink=.8)
    label_panels(axes)
    fig.tight_layout();save_eda_figure(fig,number)

# ED01: trip-count distribution.
fig,ax=plt.subplots(figsize=(10,4.2))
counts=train['nhhldtrip'].clip(upper=25).value_counts().sort_index()
ax.bar(counts.index,counts.values,color=BLUE)
ax.set(xlabel='Recorded weekday trips (25+ grouped)',ylabel='Households')
fig.tight_layout();save_eda_figure(fig,1)

# ED02: eight predictor distributions.
fig,axes=plt.subplots(2,4,figsize=(14,6.8))
for ax,field in zip(axes.flat,count_fields):
    cap=display_caps[field]
    counts=train[field].clip(upper=cap).value_counts().sort_index()
    ax.bar(counts.index.astype(str),counts.values,color=TEAL)
    labels=[f'{v}+' if v==cap else str(v) for v in counts.index]
    ax.set_xticks(range(len(labels)),labels)
    ax.set(title=FIELD_LABELS[field],xlabel=FIELD_LABELS[field],ylabel='Households')
label_panels(axes)
fig.tight_layout();save_eda_figure(fig,2)

plot_rates(['nperson','nvehicle','nlicence','workers',
            'nempft','nemppt','nemphome','nstudent'],3,ncols=4)
plot_cell_heat(size_vehicle.rename(columns={'size_band':'size','vehicle_band':'level'}),4,
               'Persons by vehicles','Number of vehicles')

# ED06: predictor Pearson correlations.
matrix=train[count_fields].corr(method='pearson')
names={'nperson':'Persons','nvehicle':'Vehicles','nlicence':'Licensed drivers',
       'nempft':'Full-time workers','nemppt':'Part-time workers',
       'nemphome':'Home workers','nstudent':'Students','workers':'Total workers'}
labels=[FIELD_LABELS[field] for field in count_fields]
fig,ax=plt.subplots(figsize=(10,8));im=ax.imshow(matrix,vmin=-1,vmax=1,cmap=CORRELATION_HEAT)
ax.set(xticks=range(len(count_fields)),xticklabels=labels,
       yticks=range(len(count_fields)),yticklabels=labels)
plt.setp(ax.get_xticklabels(),rotation=45,ha='right')
for r in range(len(count_fields)):
    for c in range(len(count_fields)):
        ax.text(c,r,f'{matrix.iloc[r,c]:.2f}',ha='center',va='center',fontsize=7)
fig.colorbar(im,ax=ax,label='Pearson correlation',shrink=.8)
fig.tight_layout();save_eda_figure(fig,6)

# ED07: joint trip frequency by size and vehicle group.
fig,axes=plt.subplots(1,2,figsize=(12,4.5))
for ax,field in zip(axes,['nperson','nvehicle']):
    block=detail_tables['TripFrequencies'].query('variable == @field')
    matrix=block.pivot(index='level',columns='trips',values='n')
    im=ax.imshow(np.log1p(matrix),aspect='auto',cmap=WARM_HEAT)
    ax.set(yticks=range(len(matrix.index)),yticklabels=matrix.index,
           xlabel='Recorded weekday household trips (20+ grouped)',ylabel=FIELD_LABELS[field])
    ax.set_xticks([0,5,10,15,20],['0','5','10','15','20+'])
    fig.colorbar(im,ax=ax,label='log(1 + households)',shrink=.8)
label_panels(axes)
fig.tight_layout();save_eda_figure(fig,7)

# ED08: Pearson correlation of each count with trips.
data=detail_tables['OutcomeCorrelations'].set_index('variable_1').reindex(count_fields)
fig,ax=plt.subplots(figsize=(9,4.8))
ax.barh([FIELD_LABELS[field] for field in data.index],data['pearson'],color=BLUE)
ax.axvline(0,color=INK,lw=.7)
ax.set(xlabel='Pearson correlation with weekday household trips',xlim=(0,.7))
fig.tight_layout();save_eda_figure(fig,8)

plot_rates(['dwelling','weekday','region'],9,ncols=3)
plot_cell_heat(size_driver.rename(columns={'size_band':'size','driver_band':'level'}),10,
               'Persons by licensed drivers','Number of licensed drivers')

# ED11: driver profiles within household-size groups, with cell support.
fig,ax=plt.subplots(figsize=(9,5))
for size,block in size_driver.groupby('size_band'):
    ax.plot(block['driver_band'],block['mean_trips'],marker='o',label=f'{size}{"+" if size==SIZE_TOP else ""} persons')
    for _,row in block.iterrows():
        if row['n'] and row['n']<MIN_CELL_N:
            ax.annotate('sparse',(row['driver_band'],row['mean_trips']),fontsize=7)
ax.set(xticks=range(DRIVER_TOP+1),xticklabels=['0','1','2','3+'],
       xlabel='Number of licensed drivers',ylabel='Mean weekday household trips')
ax.legend(title='Household size',fontsize=8)
fig.tight_layout();save_eda_figure(fig,11)
# S1: raw vehicle frequency review before eligibility screening, training assignment only.
fig,axes=plt.subplots(1,2,figsize=(12,4.5))
counts=detail_tables['VehiclePreExclusion'].set_index('nvehicle')['training_households']
main=[int(counts.get(k,0)) for k in range(10)]+[int(counts.loc[counts.index>=10].sum())]
axes[0].bar(range(11),main,color=BLUE)
axes[0].set(xticks=range(11),xticklabels=[str(k) for k in range(10)]+['10+'],
            xlabel='Number of vehicles',ylabel='Households')
tail=counts.loc[counts.index>=5]
axes[1].bar(tail.index.astype(str),tail.values,color=GOLD)
axes[1].tick_params(axis='x',labelrotation=60)
axes[1].set(xlabel='Number of vehicles',ylabel='Households')
label_panels(axes)
fig.tight_layout()
fig.savefig(FIGURES/'S1_vehicle_frequencies.png',dpi=240,facecolor='white',bbox_inches='tight')
plt.close(fig)

# RV01: companion characteristics of 5+ vehicle households before exclusions.
fig,axes=plt.subplots(2,2,figsize=(11,7.5))
for ax,field,title in zip(axes.flat[:3],['nperson','nlicence','nhhldtrip'],
                          ['Household size','Licensed drivers','Recorded weekday trips']):
    for band,color in [('5-9',BLUE),('10+',GOLD)]:
        block=vehicle_tail.loc[vehicle_tail['vehicle_band'].eq(band)]
        ax.scatter(block['nvehicle'],block[field],s=24,alpha=.65,color=color,
                   label=f'{band} vehicles (n={len(block)})')
    ax.set(xscale='log',xlabel='Recorded vehicles (log scale)',ylabel=title,title=title)
    ax.set_xticks([5,10,20,40,99],['5','10','20','40','99'])
axes[0,0].legend(fontsize=8)
for band,color in [('5-9',BLUE),('10+',GOLD)]:
    block=vehicle_tail.loc[vehicle_tail['vehicle_band'].eq(band)]
    shares=block['dwelltype'].value_counts(normalize=True).reindex([1,2,3,9],fill_value=0)
    axes[1,1].bar(np.arange(4)+(-.18 if band=='5-9' else .18),100*shares,
                  width=.35,label=band,color=color)
axes[1,1].set(xticks=range(4),xticklabels=['House','Apartment','Townhouse','Unknown'],
              ylabel='Within-band share (%)',title='Dwelling type')
axes[1,1].legend(fontsize=8)
label_panels(axes)
fig.tight_layout()
fig.savefig(FIGURES/'Figure_RV01_vehicle_household_review.png',dpi=240,facecolor='white',bbox_inches='tight')
plt.close(fig)
print('Saved 13 restored training-only figures plus 3 current EDA figures.')

Saved 13 restored training-only figures plus 3 current EDA figures.


In [8]:

derived_rows = []
def derive(quantity, value, unit, reference, sample, source):
    derived_rows.append(dict(quantity=quantity, value=value, unit=unit,
        reference_or_denominator=reference, sample=sample, source=source))
centers = {'nperson':3, 'nvehicle':1, 'nlicence':2}
descriptive_rows = []
for field in ['nhhldtrip','nperson','nvehicle','nlicence','nempft','nemppt','nemphome','nstudent','expf']:
    x = train[field]
    descriptive_rows.append(dict(panel='A: counts and expansion factor', variable=field,
        mean=x.mean(), SD=x.std(ddof=1), median=x.median(), mode=', '.join(map(str,x.mode().tolist())),
        minimum=x.min(), maximum=x.max(), percent_zero=np.nan if field=='expf' else 100*x.eq(0).mean(),
        weighted_mean=np.nan if field=='expf' else np.average(x,weights=train['expf']), centering_value=centers.get(field,np.nan)))
for field in ['dwelling','weekday','region']:
    for level,count in train[field].value_counts(sort=False).items():
        descriptive_rows.append(dict(panel='B: categories',variable=field,level=level,
            records=count,percent=100*count/len(train)))
descriptive = pd.DataFrame(descriptive_rows)
possible = size_driver.loc[size_driver.structurally_possible]
assert len(possible)==17 and possible.n.sum()==len(train) and possible.sd_trips.notna().all()
for stat in ['min','max']:
    derive(f'Cell CV {stat}',getattr(possible.cv,stat)(),'ratio','cell mean trips','training','SizeDriverCells: sample SD / mean')
rates = size_driver.pivot(index='size_band',columns='driver_band',values='mean_trips')
for low,high,sizes in [(0,2,range(2,6)),(2,3,range(3,6))]:
    for size in sizes:
        derive(f'Licence increment {low} to {high if high<3 else "3+"}; persons {size if size<5 else "5+"}',
            rates.loc[size,high]-rates.loc[size,low],'trips','lower licence group','training','SizeDriverCells: higher minus lower group mean')
assert zone_support.households.sum()==len(train)
derive('Training zones',len(zone_support),'zones','eligible training','training','ZoneSupport: row count')
for stat in ['median','min','max']:
    derive(f'Training households per zone: {stat}',getattr(zone_support.households,stat)(),'households','zone','training','ZoneSupport.households')
for row in weighting.itertuples(index=False):
    scale=100 if 'share' in row.measure else 1
    derive(f'{row.measure}: weighted minus unweighted',scale*(row.expansion_weighted-row.unweighted),
        'percentage points' if scale==100 else 'trips','unweighted statistic','training','Weighting: weighted minus unweighted; shares multiplied by 100')
for lower in [5,8]:
    derive(f'Vehicle tail {lower}-9',int(train.nvehicle.between(lower,9).sum()),'households','eligible training','training','Count nvehicle within inclusive bounds')
derived_eda = pd.DataFrame(derived_rows)
print('Derived quantities'); display(derived_eda); display(descriptive)

dictionary = pd.DataFrame([
    {'field':'nhhldtrip','meaning':'Recorded household trips on survey weekday; outcome covers ages 11+'},
    {'field':'nperson','meaning':'Household residents; includes ages below trip-recording threshold'},
    {'field':'nvehicle','meaning':'Vehicles available; 2006 guide range 0-99; counts 10+ excluded as unusual'},
    {'field':'nlicence','meaning':'Licensed drivers in household'},
    {'field':'nempft / nemppt','meaning':'Full-time / part-time workers; work-at-home persons already included'},
    {'field':'nstudent','meaning':'Students; children aged 6-10 are assumed full-time students'},
    {'field':'dwelltype','meaning':'1 House, 2 Apartment, 3 Townhouse, 9 Unknown'},
    {'field':'tripday','meaning':'Survey weekday, Monday through Friday'},
    {'field':'expf','meaning':'Household expansion factor; not a substantive predictor'},
    {'field':'gta01hhld','meaning':'Residential traffic zone (2001 zone numbering)'},
])
final_tables = {
    'SampleFlow':flow,'DataDictionary':dictionary,'NumericSummary':numeric,
    'TripDistribution':trip_distribution,'GroupTripRates':categories,
    'SizeDriverCells':size_driver,'SizeVehicleCells':size_vehicle,
    'ZoneSupport':zone_support,'Correlations':correlations,'Weighting':weighting,
    **detail_tables, 'DescriptiveStatistics':descriptive, 'Derived_EDA':derived_eda,
}
contents = pd.DataFrame([{'sheet':name,'rows':len(frame),'columns':len(frame.columns)}
                         for name,frame in final_tables.items()])
write_workbook(FINAL_BOOK,{'Contents':contents,**final_tables})
split_policy = pd.DataFrame([{'seed':SPLIT_SEED,'test_fraction':TEST_FRACTION,
    'source_households':source_n,'test_assigned_before_exclusions':int(split['split'].eq('test').sum()),
    'numpy_version':np.__version__, 'python_version':sys.version.split()[0],
    'python_executable':sys.executable,
    'method':'Sort household IDs; NumPy PCG64 permutation; first rounded 30% assigned to test'}])
write_workbook(TRAIN_BOOK,{'Split':split,'SplitPolicy':split_policy,'Train':train,
    'Exclusions':excluded,'QualityChecks':quality})

assert FINAL_BOOK.exists() and TRAIN_BOOK.exists() and TEST_BOOK.exists()
assert contents['rows'].sum()>0 and size_driver['n'].sum()==train_n
print(f'Saved {len(final_tables)} final EDA tables, 15 figures, and separate training/test workbooks.')
display(contents)
# Paper figures use the same eligible training summaries as the EDA workbook.
OUT = ROOT / 'paper' / 'figures'
OUT.mkdir(parents=True, exist_ok=True)
plt.rcParams['font.size'] = 9
trip = final_tables['TripDistribution']
counts = trip.groupby(trip['trips'].clip(upper=25))['households'].sum().reindex(range(26), fill_value=0)
n = int(counts.sum())
fig, ax = plt.subplots(figsize=(7, 2.55), layout='constrained')
ax.bar(counts.index, 100 * counts / n, color=BLUE, width=.8)
ax.set(xlabel='Recorded weekday household trips', ylabel='Households (%)',
       xticks=[0,5,10,15,20,25], xticklabels=['0','5','10','15','20','25+'])
ax.set_ylim(bottom=0)
fig.savefig(OUT / 'eda_trip_distribution.pdf', bbox_inches='tight')
plt.close(fig)

rates = final_tables['DisplayGroupRates']
fig, axes = plt.subplots(2, 2, figsize=(7, 4.25), sharey=True, layout='constrained')
for ax, field, title in zip(axes.flat, ['nperson','nvehicle','nempft','nstudent'],
                            ['(a) Household size','(b) Vehicles','(c) Full-time workers','(d) Students']):
    block = rates.loc[rates.variable.eq(field)]
    assert int(block.n.sum()) == n
    ax.plot(range(len(block)), block.mean_trips, marker='o', color=BLUE, lw=1.5, ms=4)
    ax.set(xticks=range(len(block)), xticklabels=block.level_label, title=title, ylim=(0,14))
    ax.grid(axis='y', alpha=.2)
for ax in axes[:,0]:
    ax.set_ylabel('Mean weekday trips')
fig.savefig(OUT / 'eda_predictor_rates.pdf', bbox_inches='tight')
plt.close(fig)

cells = final_tables['SizeDriverCells']
assert int(cells.n.sum()) == n
mean = cells.pivot(index='size_band', columns='driver_band', values='mean_trips')
support = cells.pivot(index='size_band', columns='driver_band', values='n')
possible = cells.pivot(index='size_band', columns='driver_band', values='structurally_possible')
masked = np.ma.masked_where(~possible.to_numpy() | support.eq(0).to_numpy(), mean.to_numpy())
fig, ax = plt.subplots(figsize=(7, 3.85), layout='constrained')
im = ax.imshow(masked, cmap=WARM_HEAT, aspect='auto')
ax.set(xticks=range(4), xticklabels=['0','1','2','3+'], yticks=range(5),
       yticklabels=['1','2','3','4','5+'], xlabel='Number of licensed drivers', ylabel='Household size')
for r in range(5):
    for c in range(4):
        if possible.iat[r,c] and support.iat[r,c]:
            rate = mean.iat[r,c]
            ax.text(c,r,f'{rate:.2f}\n(n={support.iat[r,c]:,})',ha='center',va='center',
                    fontsize=8,color='white' if im.norm(rate)>.68 else INK)
fig.colorbar(im, ax=ax, label='Mean weekday trips', shrink=.85)
fig.savefig(OUT / 'eda_size_driver_cells.pdf', bbox_inches='tight')
plt.close(fig)

# Display linear associations for the outcome and the distinct household counts.
# Total workers is omitted because it is the sum of the two worker counts.
fields = ['nhhldtrip', 'nperson', 'nvehicle', 'nlicence',
          'nempft', 'nemppt', 'nstudent', 'nemphome']
labels = ['Household trips', 'Household size', 'Vehicles', 'Licensed\ndrivers',
          'Full-time\nworkers', 'Part-time\nworkers', 'Students', 'Home\nworkers']
pairwise = final_tables['PairwiseCorrelations']
outcome = final_tables['OutcomeCorrelations']
assert (pairwise.n == n).all() and (outcome.n == n).all()
corr = pd.DataFrame(np.nan, index=fields, columns=fields)
for field in fields:
    corr.loc[field, field] = 1.0
for row in pd.concat([pairwise, outcome]).itertuples(index=False):
    if row.variable_1 in fields and row.variable_2 in fields:
        corr.loc[row.variable_1, row.variable_2] = row.pearson
        corr.loc[row.variable_2, row.variable_1] = row.pearson
assert corr.notna().all().all()
assert np.allclose(corr, corr.T)
fig, ax = plt.subplots(figsize=(7, 5.35), layout='constrained')
image = ax.imshow(corr.to_numpy(), cmap=CORRELATION_HEAT,
                  vmin=-1, vmax=1, aspect='equal')
ax.set(xticks=range(len(fields)), yticks=range(len(fields)),
       xticklabels=labels, yticklabels=labels)
ax.tick_params(axis='x', labelrotation=45, length=0)
ax.tick_params(axis='y', length=0)
for label in ax.get_xticklabels():
    label.set_ha('right')
for i in range(len(fields)):
    for j in range(len(fields)):
        value = corr.iat[i, j]
        ax.text(j, i, f'{value:.2f}', ha='center', va='center', fontsize=8,
                color='white' if value >= .88 or value <= -.88 else INK)
fig.colorbar(image, ax=ax, label='Pearson correlation', shrink=.78)
fig.savefig(OUT / 'eda_correlation_heatmap.pdf', bbox_inches='tight')
plt.close(fig)
print(f'Saved four paper figures from {n:,} training households to {OUT}')
plt.rcParams['font.size'] = 10


Derived quantities


,quantity,value,unit,reference_or_denominator,sample,source
0,Cell CV min,0.474701,ratio,cell mean trips,training,SizeDriverCells: sample SD / mean
1,Cell CV max,1.219056,ratio,cell mean trips,training,SizeDriverCells: sample SD / mean
2,Licence increment 0 to 2; persons 2,2.718964,trips,lower licence group,training,SizeDriverCells: higher minus lower group mean
3,Licence increment 0 to 2; persons 3,2.446098,trips,lower licence group,training,SizeDriverCells: higher minus lower group mean
4,Licence increment 0 to 2; persons 4,3.265029,trips,lower licence group,training,SizeDriverCells: higher minus lower group mean
5,Licence increment 0 to 2; persons 5+,2.755778,trips,lower licence group,training,SizeDriverCells: higher minus lower group mean
6,Licence increment 2 to 3+; persons 3,0.871530,trips,lower licence group,training,SizeDriverCells: higher minus lower group mean
7,Licence increment 2 to 3+; persons 4,1.664933,trips,lower licence group,training,SizeDriverCells: higher minus lower group mean
8,Licence increment 2 to 3+; persons 5+,2.648621,trips,lower licence group,training,SizeDriverCells: higher minus lower group mean
9,Training zones,1158.000000,zones,eligible training,training,ZoneSupport: row count


,panel,variable,mean,SD,median,mode,minimum,maximum,percent_zero,weighted_mean,centering_value,level,records,percent
0,A: counts and expansion factor,nhhldtrip,5.642364,4.548931,4.00,4,0.00,55.00,11.141575,5.638513,NaN,NaN,NaN,NaN
1,A: counts and expansion factor,nperson,2.736915,1.417725,2.00,2,1.00,16.00,0.000000,2.738821,3.0,NaN,NaN,NaN
2,A: counts and expansion factor,nvehicle,1.376430,0.929003,1.00,1,0.00,9.00,16.401428,1.373286,1.0,NaN,NaN,NaN
3,A: counts and expansion factor,nlicence,1.729977,0.983994,2.00,2,0.00,9.00,9.589964,1.729256,2.0,NaN,NaN,NaN
4,A: counts and expansion factor,nempft,1.112993,0.926035,1.00,1,0.00,7.00,29.886501,1.116922,NaN,NaN,NaN,NaN
5,A: counts and expansion factor,nemppt,0.237413,0.510708,0.00,0,0.00,5.00,79.786175,0.237117,NaN,NaN,NaN,NaN
6,A: counts and expansion factor,nemphome,0.109149,0.356132,0.00,0,0.00,7.00,90.459050,0.107757,NaN,NaN,NaN,NaN
7,A: counts and expansion factor,nstudent,0.633281,0.951703,0.00,0,0.00,9.00,62.009282,0.635266,NaN,NaN,NaN,NaN
8,A: counts and expansion factor,expf,19.391051,2.514385,19.35,20.71,10.83,28.06,NaN,NaN,NaN,NaN,NaN,NaN
9,B: categories,dwelling,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,House,40707.0,62.350851


Saved 25 final EDA tables, 15 figures, and separate training/test workbooks.


,sheet,rows,columns
0,SampleFlow,6,2
1,DataDictionary,10,2
2,NumericSummary,10,10
3,TripDistribution,47,3
4,GroupTripRates,45,6
5,SizeDriverCells,20,10
6,SizeVehicleCells,20,9
7,ZoneSupport,1158,4
8,Correlations,9,10
9,Weighting,2,3


Saved four paper figures from 65,287 training households to /home/mst_lab/Desktop/king/Classwork/TRP630/Term_paper/trp631termpaper/paper/figures
